# Multi-LLM-as-Judge: Medical AI Benchmark
## Complete SMU SuperPOD Experiment Notebook

**Tailored to:** `mkotha` on SMU SuperPOD | Account: `xnluo_ai_chatbot_cognitive_0002`

**Dataset:** `benchmark_dataset/source_datasets/benchmark_dataset_500.csv`
525 rows | 5 domains (Cardiology, Emergency, Neurology, Pediatrics, Pharmacology) | 105 per domain

| Phase | Cell | What happens |
|---|---|---|
| 0 | — | SSH + SOCKS proxy + tmux + srun (local terminal) |
| 1 | 1 | Verify GPU node, venv, paths |
| 2 | 2 | Install deps |
| 3 | 3 | Load .env + HF cache |
| 3 | 4 | Download all 4 judge models |
| 4 | 5 | Kill any old vLLM + launch 4 fresh servers |
| 4 | 6 | Tail logs (optional check) |
| 5 | 7 | Health check all 4 judges (20 min timeout) |
| 6 | 8 | Repo structure check |
| 6 | 9 | Unit tests |
| 7 | 10 | Exp 1 — Dataset analysis |
| 8 | 11 | Exp 2 — Agreement analysis |
| 9 | 12 | Exp 3 — Rubric sensitivity |
| 10 | 13 | Exp 4 — Box plots |
| 11 | 14 | Results summary + CSV |
| 12 | 15 | Display figures inline |
| 13 | 16 | Push to GitHub |
| 14 | 17 | Cleanup / kill vLLM |

---

## Phase 0 — Terminal Commands (run BEFORE opening this notebook)

```bash
# 1. On your LOCAL machine:
ssh -C -D 8080 mkotha@superpod.smu.edu

# 2. On SuperPOD login node:
tmux new -s cs7325

# 3. Request GPU node (inside tmux):
srun -A xnluo_ai_chatbot_cognitive_0002 -N1 -G2 -c32 --mem=192G --time=4:00:00 --pty $SHELL

# 4. On the GPU compute node:
cd /lustre/smuexa01/client/users/mkotha/CS7325/Multi_LLM_as_Judge_Medical_AI
git pull origin main
```


In [ ]:
# CELL 1: Verify GPU, venv, paths
import subprocess, os, sys, importlib, json
from pathlib import Path
from datetime import datetime
import pandas as pd

def shell(cmd, check=False):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (result.stdout + result.stderr).strip()
    if out: print(out)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}')
    return result.returncode

LUSTRE_BASE  = Path('/lustre/smuexa01/client/users/mkotha/CS7325')
ROOT         = LUSTRE_BASE / 'Multi_LLM_as_Judge_Medical_AI'
VENV         = LUSTRE_BASE / '.venv'
HF_CACHE     = LUSTRE_BASE / 'hf_models'
LOG_DIR      = LUSTRE_BASE / 'vllm_logs'
PYTHON       = str(VENV / 'bin' / 'python')
DATASET_PATH = ROOT / 'benchmark_dataset' / 'source_datasets' / 'benchmark_dataset_500.csv'

HF_CACHE.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Repo root    : {ROOT}')
print(f'Dataset      : {DATASET_PATH}')
print(f'Python       : {sys.executable}')
print(f'Started      : {datetime.now().isoformat()}')

# Verify dataset
if DATASET_PATH.exists():
    df_check = pd.read_csv(DATASET_PATH)
    print(f'\n=== Dataset loaded: {len(df_check)} rows ===')
    print(df_check['domain'].value_counts().sort_index().to_string())
else:
    raise FileNotFoundError(f'Dataset not found: {DATASET_PATH}\nRun: git pull origin main')

print('\n=== GPU Status ===')
shell('nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv,noheader')


In [ ]:
# CELL 2: Install all deps
REQUIREMENTS = ROOT / 'requirements.txt'
print(f'Installing from: {REQUIREMENTS}')
shell(f'{sys.executable} -m pip install -r {REQUIREMENTS} -q')
shell(f'{sys.executable} -m pip install vllm tabulate ipywidgets -q')

REQUIRED = ['vllm','httpx','datasets','huggingface_hub','pandas','matplotlib','seaborn','dotenv','pytest','tabulate']
print('\n=== Package check ===')
all_ok = True
for pkg in REQUIRED:
    try:
        m = importlib.import_module(pkg.replace('-','_'))
        print(f'  ✅ {pkg:<20} {getattr(m,"__version__","?")}'  )
    except ImportError:
        print(f'  ❌ {pkg} MISSING'); all_ok = False
print('\n✅ All OK' if all_ok else '\n❌ Re-run this cell')


In [ ]:
# CELL 3: Load .env + set HF cache
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')
os.environ['HF_HOME']            = str(HF_CACHE)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE']  = str(HF_CACHE / 'datasets')
HF_TOKEN = os.environ.get('HF_TOKEN')
GH_TOKEN = os.environ.get('GITHUB_TOKEN')
print(f'HF_TOKEN:     {"✅ set" if HF_TOKEN else "❌ NOT SET"}')
print(f'GITHUB_TOKEN: {"✅ set" if GH_TOKEN else "⚠️  not set"}')
shell(f'du -sh {HF_CACHE} 2>/dev/null || echo empty')


In [ ]:
# CELL 4: Download all judge models (skips already-cached)
from huggingface_hub import snapshot_download
import time
JUDGE_MODELS = [
    {'id': 'medgemma',   'hf_id': 'google/medgemma-4b-it',     'needs_token': True},
    {'id': 'biomistral', 'hf_id': 'BioMistral/BioMistral-7B',  'needs_token': False},
    {'id': 'meditron',   'hf_id': 'epfl-llm/meditron-7b',      'needs_token': False},
    {'id': 'medalpaca',  'hf_id': 'medalpaca/medalpaca-7b',     'needs_token': False},
]
for m in JUDGE_MODELS:
    marker = HF_CACHE / f'models--{m["hf_id"].replace("/","--")}'
    if marker.exists() and any(marker.rglob('config.json')):
        print(f'✅ [{m["id"]:12s}] cached'); continue
    print(f'⏬  Downloading {m["hf_id"]}...')
    t0 = time.time()
    try:
        snapshot_download(repo_id=m['hf_id'], cache_dir=str(HF_CACHE),
            token=HF_TOKEN if m['needs_token'] else None,
            ignore_patterns=['*.msgpack','*.h5','flax_model*','tf_model*'])
        print(f'  ✅ Done {(time.time()-t0)/60:.1f} min')
    except Exception as e:
        print(f'  ❌ {e}')


In [ ]:
# CELL 5: Kill old vLLM + launch 4 fresh servers
import time, subprocess
subprocess.run('pkill -9 -f "vllm.entrypoints" 2>/dev/null || true', shell=True)
time.sleep(5)
print('GPU after cleanup:'); shell('nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader')

def get_snapshot_path(hf_id):
    snap_dir = HF_CACHE / f'models--{hf_id.replace("/","--")}' / 'snapshots'
    snaps = list(snap_dir.iterdir()) if snap_dir.exists() else []
    if not snaps: raise FileNotFoundError(f'Not cached: {hf_id} — run Cell 4')
    return str(sorted(snaps, key=lambda p: p.stat().st_mtime)[-1])

JUDGE_CONFIGS = [
    {'id':'medgemma',  'hf_id':'google/medgemma-4b-it',    'port':8001,'gpu':'0','max_len':4096,'gpu_mem':0.20,'extra':'--trust-remote-code'},
    {'id':'biomistral','hf_id':'BioMistral/BioMistral-7B', 'port':8002,'gpu':'1','max_len':4096,'gpu_mem':0.50,'extra':''},
    {'id':'meditron',  'hf_id':'epfl-llm/meditron-7b',     'port':8003,'gpu':'0','max_len':2048,'gpu_mem':0.65,'extra':''},
    {'id':'medalpaca', 'hf_id':'medalpaca/medalpaca-7b',    'port':8004,'gpu':'1','max_len':2048,'gpu_mem':0.42,'extra':''},
]
snapshot_paths = {j['id']: get_snapshot_path(j['hf_id']) for j in JUDGE_CONFIGS}

def wait_healthy(port, name, timeout=600):
    import urllib.request
    t0 = time.time()
    while time.time()-t0 < timeout:
        try:
            with urllib.request.urlopen(f'http://localhost:{port}/health', timeout=5) as r:
                if r.status == 200: return True
        except: pass
        time.sleep(10); print(f'  ⏳ [{name}] {time.time()-t0:.0f}s', end='\r')
    return False

procs = {}
for j in JUDGE_CONFIGS:
    print(f'\n--- Launching {j["id"]} (GPU {j["gpu"]}, port {j["port"]}) ---')
    subprocess.run(f'fuser -k {j["port"]}/tcp 2>/dev/null || true', shell=True); time.sleep(2)
    log = str(LOG_DIR / f'vllm_{j["id"]}.log')
    cmd = (f'CUDA_VISIBLE_DEVICES={j["gpu"]} HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 HF_HOME={HF_CACHE} '
           f'{PYTHON} -m vllm.entrypoints.openai.api_server '
           f'--model {snapshot_paths[j["id"]]} --port {j["port"]} --host 0.0.0.0 '
           f'--gpu-memory-utilization {j["gpu_mem"]} --max-model-len {j["max_len"]} '
           f'--dtype bfloat16 {j["extra"]} > {log} 2>&1')
    proc = subprocess.Popen(cmd, shell=True)
    print(f'  🚀 PID={proc.pid}')
    ok = wait_healthy(j['port'], j['id'])
    if ok:
        print(f'  ✅ [{j["id"]}] UP on port {j["port"]}')
        procs[j['id']] = proc
    else:
        subprocess.run(f'tail -15 {log}', shell=True)
        raise RuntimeError(f'Judge {j["id"]} failed — fix above then re-run this cell')

JUDGE_URLS = {j['id']: f'http://localhost:{j["port"]}' for j in JUDGE_CONFIGS}
print('\n✅ ALL 4 JUDGES LAUNCHED')


In [ ]:
# CELL 6 (optional): Tail vLLM logs
for j in JUDGE_CONFIGS:
    print(f'\n--- [{j["id"]}] ---')
    shell(f'tail -8 {LOG_DIR}/vllm_{j["id"]}.log 2>/dev/null || echo "(no log)"')


In [ ]:
# CELL 7: Health check all 4 judges
import httpx, time
if 'JUDGE_URLS' not in dir():
    JUDGE_URLS = {'medgemma':'http://localhost:8001','biomistral':'http://localhost:8002',
                  'meditron':'http://localhost:8003','medalpaca':'http://localhost:8004'}
ready = {k: False for k in JUDGE_URLS}; t0 = time.time()
while not all(ready.values()) and time.time()-t0 < 1200:
    for name, url in JUDGE_URLS.items():
        if ready[name]: continue
        try:
            if httpx.get(f'{url}/health', timeout=4).status_code == 200:
                ready[name] = True; print(f'  ✅ {name} UP [{time.time()-t0:.0f}s]')
        except: pass
    if not all(ready.values()):
        print(f'  ⏳ Pending: {[k for k,v in ready.items() if not v]} [{time.time()-t0:.0f}s]')
        time.sleep(30)
if all(ready.values()): print('\n✅ ALL 4 JUDGES READY — proceed to experiments')
else: print(f'\n❌ Failed: {[k for k,v in ready.items() if not v]}')


In [ ]:
# CELL 8: Repo structure check
REQUIRED_PATHS = [
    ROOT/'core'/'wrapper.py', ROOT/'core'/'agreement.py',
    ROOT/'core'/'rubric_engine.py', ROOT/'core'/'metrics.py',
    ROOT/'core'/'schemas.py',
    ROOT/'rubrics'/'rubric1_pemat.json', ROOT/'rubrics'/'rubric2_healthbench.json',
    ROOT/'rubrics'/'rubric3_clinical_eval.json', ROOT/'rubrics'/'rubric4_prometheus.json',
    ROOT/'experiments'/'exp1_dataset_analysis.py',
    ROOT/'experiments'/'exp2_agreement_analysis.py',
    ROOT/'experiments'/'exp3_rubric_sensitivity.py',
    ROOT/'experiments'/'exp4_boxplot_agreement.py',
    DATASET_PATH,
]
print('=== Repo structure check ===')
all_ok = True
for p in REQUIRED_PATHS:
    ok = p.exists(); all_ok = all_ok and ok
    print(f'  {"✅" if ok else "❌"} {p.relative_to(ROOT)}')
print('\n✅ All files present' if all_ok else f'\n❌ Missing files — run: git -C {ROOT} pull origin main')


In [ ]:
# CELL 9: Unit tests
rc = shell(f'{sys.executable} -m pytest {ROOT}/tests/test_core.py -v --tb=short')
print('\n✅ All tests passed' if rc == 0 else '\n⚠️  Tests failed')


In [ ]:
# CELL 10: Experiment 1 — Dataset Analysis
import importlib, experiments.exp1_dataset_analysis as exp1_mod
importlib.reload(exp1_mod)

print('Running Experiment 1: Dataset Analysis')
print('='*60)
exp1_mod.main()

# Show balanced domain breakdown from new dataset
df_bench = pd.read_csv(DATASET_PATH)
counts = df_bench['domain'].value_counts().sort_index()
df_cat = counts.reset_index()
df_cat.columns = ['Domain', 'N']
df_cat.loc[len(df_cat)] = ['TOTAL', df_cat['N'].sum()]
print('\n=== Dataset Breakdown (benchmark_dataset_500.csv) ===')
print(df_cat.to_markdown(index=False))
print(f'\nSources: {df_bench["source"].value_counts().to_dict()}')
print(f'\n✅ Exp1 complete — {len(df_bench)} questions across {df_bench["domain"].nunique()} domains')


In [ ]:
# CELL 11: Experiment 2 — Agreement Analysis
import experiments.exp2_agreement_analysis as exp2_mod
importlib.reload(exp2_mod)

print('Running Experiment 2: Per-Rubric Agreement Analysis')
print('='*60)
exp2_mod.main()

from collections import Counter
exp2_path = ROOT / 'results' / 'exp2_agreement_results.json'
with open(exp2_path) as f: exp2_data = json.load(f)

rows = []
for block in exp2_data:
    results = block['results']; total = len(results)
    classes = Counter(r['agreement_class'] for r in results)
    mean_pw = sum(r['mean_pairwise_agreement'] for r in results) / total
    rows.append({'Rubric': block['rubric_name'].split('—')[0].strip()[:30],
                 'N': total,
                 'Fully Agree%': round(classes.get('fully_agree',0)/total*100,1),
                 'Majority%': round(classes.get('majority_agree',0)/total*100,1),
                 'Split%': round(classes.get('split',0)/total*100,1),
                 'Disagree%': round(classes.get('full_disagree',0)/total*100,1),
                 'Mean PW%': round(mean_pw,1)})
df_exp2 = pd.DataFrame(rows)
print('\n=== Agreement Table ===')
print(df_exp2.to_markdown(index=False))
print('\n✅ Exp2 complete')


In [ ]:
# CELL 12: Experiment 3 — Rubric Sensitivity
import experiments.exp3_rubric_sensitivity as exp3_mod
importlib.reload(exp3_mod)
print('Running Experiment 3: Rubric Sensitivity'); print('='*60)
exp3_mod.main()
exp3_path = ROOT / 'results' / 'exp3_sensitivity_results.json'
with open(exp3_path) as f: exp3_data = json.load(f)
sens_rows = [{'Rubric': b['rubric_name'].split('—')[0].strip()[:25],
              'Variant': v['scoring_variant'],
              'Mean%': v['mean_pairwise_agreement'],
              'Std%': v['std_pairwise_agreement']}
             for b in exp3_data for v in b['variants']]
print(pd.DataFrame(sens_rows).to_markdown(index=False))
print('\n✅ Exp3 complete')


In [ ]:
# CELL 13: Experiment 4 — Box Plots
import experiments.exp4_boxplot_agreement as exp4_mod
importlib.reload(exp4_mod)
print('Running Experiment 4: Box Plot Generation'); print('='*60)
exp4_mod.main()
pngs = sorted((ROOT / 'results' / 'figures').glob('*.png'))
print(f'\n✅ Exp4 complete: {len(pngs)} figures')
for p in pngs: print(f'  📊 {p.name}  ({p.stat().st_size//1024} KB)')


In [ ]:
# CELL 14: Full results summary + save CSV
print('=== FINAL RESULTS ===\n')
print('[Exp 1] Dataset (benchmark_dataset_500.csv):')
print(counts.to_string())
print(f'TOTAL: {counts.sum()}')
print('\n[Exp 2] Agreement Table:')
print(df_exp2.to_markdown(index=False))
print('\n[Exp 3] Best scoring variant per rubric:')
for b in exp3_data:
    best = max(b['variants'], key=lambda v: v['mean_pairwise_agreement'])
    print(f'  {b["rubric_name"][:30]:30s} → {best["scoring_variant"]} ({best["mean_pairwise_agreement"]:.1f}%)')
print(f'\n[Exp 4] {len(pngs)} PNG figures in results/figures/')
df_exp2.to_csv(ROOT / 'results' / 'agreement_summary_table.csv', index=False)
print('\n✅ agreement_summary_table.csv saved')


In [ ]:
# CELL 15: Display all PNG figures inline
from IPython.display import Image, display, Markdown
for png in sorted((ROOT / 'results' / 'figures').glob('*.png')):
    display(Markdown(f'### {png.stem}'))
    display(Image(filename=str(png), width=950))


In [ ]:
# CELL 16: Commit and push results
if GH_TOKEN:
    shell(f'git -C {ROOT} remote set-url origin https://{GH_TOKEN}@github.com/m22oct2000/Multi_LLM_as_Judge_Medical_AI.git')
COMMIT_MSG = f'HPC results (SuperPOD mkotha): all 4 experiments — {datetime.now().strftime("%Y-%m-%d %H:%M")}'
shell(f'git -C {ROOT} status --short')
shell(f'git -C {ROOT} add results/')
shell(f'git -C {ROOT} add benchmark_dataset/')
shell(f'git -C {ROOT} commit -m "{COMMIT_MSG}"')
rc = shell(f'git -C {ROOT} push origin main')
if rc == 0: print('\n✅ Pushed! → https://github.com/m22oct2000/Multi_LLM_as_Judge_Medical_AI/tree/main/results')
else: print(f'\n⚠️  Push failed. cd {ROOT} && git push origin main')


In [ ]:
# CELL 17: Kill vLLM + free GPUs
if 'procs' in dir():
    for name, proc in procs.items():
        try: proc.terminate(); print(f'Terminated [{name}]')
        except: pass
subprocess.run('pkill -9 -f "vllm.entrypoints" 2>/dev/null || true', shell=True)
import time; time.sleep(4)
print('\n=== GPU after cleanup ===')
shell('nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv,noheader')
print('\n✅ Done. GPUs free.')
